In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = pd.Series(data.target)
print(df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(["target"], axis=1), df["target"], test_size=0.2, random_state=101
)

print(
    "* Train set:",
    X_train.shape,
    y_train.shape,
    "\n* Test set:",
    X_test.shape,
    y_test.shape,
)

In [ ]:
from sklearn.pipeline import Pipeline

### Feat Scaling
from sklearn.preprocessing import StandardScaler

### Feat Selection
from sklearn.feature_selection import SelectFromModel

### ML algorithms 
from sklearn.linear_model import LogisticRegression

def pipeline_logistic_regression():
    """
    Create a pipeline for logistic regression.

    Returns:
        pipeline (Pipeline): A pipeline object that consists of feature scaling, feature selection, and logistic regression model.
    """
    pipeline = Pipeline(
        [
            ("feat_scaling", StandardScaler()),
            ("feat_selection", SelectFromModel(LogisticRegression(random_state=101))),
            ("model", LogisticRegression(random_state=101)),
        ]
    )

    return pipeline

In [ ]:
pipeline = pipeline_logistic_regression()
pipeline.fit(X_train, y_train)

In [ ]:
def logistic_regression_coef(model, columns):
    """
    Prints the coefficients of a logistic regression model.

    Parameters:
    - model: The trained logistic regression model.
    - columns: The column names corresponding to the coefficients.

    Returns:
    None
    """
    coeff_df = pd.DataFrame(
        model.coef_, index=["Coefficient"], columns=columns
    ).T.sort_values(["Coefficient"], key=abs, ascending=False)
    print(coeff_df)

In [ ]:
logistic_regression_coef(
    model=pipeline["model"],
    columns=X_train.columns[pipeline["feat_selection"].get_support()],
)

In [ ]:
# loads confusion_matrix and classification_report from sklearn
from sklearn.metrics import classification_report, confusion_matrix


def confusion_matrix_and_report(X, y, pipeline, label_map):
    """
    Gets features, target, pipeline, and how the levels from your target are labelled (named).
    In this case, 0 (Malignant) and 1 (Benign), so you parse a list ['Malignant' , 'Benign'].

    Args:
        X (array-like): The input features.
        y (array-like): The target values.
        pipeline (object): The trained pipeline model.
        label_map (list): The list of labels for the target values.

    Returns:
        None

    This function performs the following steps:
    - Predicts the target values based on the input features using the provided pipeline.
    - Computes and displays the confusion matrix, which compares the predicted values with the actual values.
      The predicted values are shown as rows, and the actual values are shown as columns in the matrix.
    - Displays the classification report, which provides metrics such as precision, recall, and F1-score.

    """

    prediction = pipeline.predict(X)

    print("---  Confusion Matrix  ---")
    print(
        pd.DataFrame(
            confusion_matrix(y_true=prediction, y_pred=y),
            columns=[["Actual " + sub for sub in label_map]],
            index=[["Prediction " + sub for sub in label_map]],
        )
    )
    print("\n")

    print("---  Classification Report  ---")
    print(classification_report(y, prediction, target_names=label_map), "\n")


def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    """
    Calculates and displays the performance metrics of a classification model.

    Parameters:
    - X_train (array-like): The feature matrix of the training set.
    - y_train (array-like): The target labels of the training set.
    - X_test (array-like): The feature matrix of the test set.
    - y_test (array-like): The target labels of the test set.
    - pipeline (object): The trained classification pipeline.
    - label_map (list): A list containing the labels for the target variable.

    Returns:
    None

    This function calculates and displays the confusion matrix and classification report
    for both the training set and the test set. The confusion matrix provides a summary
    of the model's performance by showing the number of true positives, true negatives,
    false positives, and false negatives. The classification report provides additional
    performance metrics such as precision, recall, and F1-score.

    Example usage:
    clf_performance(X_train, y_train, X_test, y_test, pipeline, ['Malignant', 'Benign'])
    """
    print("#### Train Set #### \n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("#### Test Set ####\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)


In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline,
    label_map=["Malignant", "Benign"],
)

In [ ]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline,
    label_map=["0", "1"],
)  # it will display the classes as 0 and 1
# but "0" and "1" should be a string